In [ ]:
!pip install -q iterative-stratification
!pip install -q iterative-stratification

In [ ]:

import os, gc, time, json, random
import numpy as np, pandas as pd, cv2
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from contextlib import nullcontext
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

def seed_everything(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
print(f"Device: {device} | GPUs: {num_gpus}", flush=True)

ROOT = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public/")
TRAIN_DIR, CSV_PATH = ROOT / "train", ROOT / "train.csv"
DEFECT_TYPES = ["defect_a", "defect_b", "defect_c", "defect_d"]
CLASS_TO_IDX = {n: i for i, n in enumerate(DEFECT_TYPES)}
CONFIG = {"seed": 42, "lr": 1e-3, "weight_decay": 1e-4,
          "class_thresholds": [0.20, 0.35, 0.575, 0.40],
          "min_defect_pixels": 255, "min_component_pixels": 75}


In [ ]:
def rle_decode(rle_string, height, width):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle_string) or not str(rle_string).strip():
        return mask.reshape((height, width), order="F")

    rle = np.asarray(str(rle_string).split(), dtype=np.int64)
    starts = rle[0::2] - 1
    lengths = rle[1::2]
    ends = starts + lengths

    if len(starts) != len(lengths):
        raise ValueError("Invalid RLE pairing.")

    for start, end in zip(starts, ends):
        mask[start:end] = 1

    return mask.reshape((height, width), order="F")


def rle_encode(mask):
    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("Mask must be a 2D array")

    pixels = (mask > 0).flatten(order="F")

    padded = np.concatenate([[0], pixels, [0]])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1

    changes[1::2] -= changes[::2]

    return " ".join(map(str, changes))

In [ ]:

df = pd.read_csv(CSV_PATH)
image_df = pd.DataFrame({"image_id": df["image_id"].drop_duplicates().values})
for d in DEFECT_TYPES:
    pos = df.loc[(df["defect_type"] == d) & df["mask_rle"].notna()
                 & (df["mask_rle"].astype(str).str.strip() != ""), "image_id"].unique()
    image_df[d] = image_df["image_id"].isin(pos).astype(int)
X = image_df["image_id"].values
y = image_df[DEFECT_TYPES].values
print(f"Images: {len(X):,} | positive rate per class: {y.mean(0).round(3)}", flush=True)


In [ ]:
# 1. Double Convolution Block
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


# 2. Encoder Block
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.down(x)


# 3. Decoder Block
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))


# 4. Baseline U-Net
class DefectUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=4):
        super().__init__()

        self.encoder = nn.Sequential(
            DoubleConv(in_channels, 32), DownBlock(32, 64), 
            DownBlock(64, 128), DownBlock(128, 256) )

        self.bottleneck = nn.Sequential( nn.MaxPool2d(2), DoubleConv(256, 512) )

        self.decoder = nn.ModuleList([
            UpBlock(512, 256, 256), UpBlock(256, 128, 128),
            UpBlock(128, 64, 64), UpBlock(64, 32, 32) ])

        self.outc = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        skips = []

        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        x = self.bottleneck(x)

        for layer, skip in zip(self.decoder, reversed(skips)):
            x = layer(x, skip)

        return self.outc(x)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(gating_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.W_x = nn.Sequential(nn.Conv2d(skip_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.psi = nn.Sequential(nn.Conv2d(inter_channels, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        attention = self.relu(g1 + x1)
        attention = self.psi(attention)
        return x * attention


class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.attention = AttentionGate(gating_channels=out_channels, skip_channels=skip_channels, inter_channels=max(out_channels // 2, 1))
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        # Match spatial dimensions
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        # Apply attention to encoder features
        skip = self.attention(x, skip)

        # Concatenate features
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class AttentionDefectUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=4):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            DoubleConv(in_channels, 32),
            DownBlock(32, 64),
            DownBlock(64, 128),
            DownBlock(128, 256)
        )

        # Bottleneck
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        # Attention Decoder
        self.decoder = nn.ModuleList([
            AttentionUpBlock(512, 256, 256),
            AttentionUpBlock(256, 128, 128),
            AttentionUpBlock(128, 64, 64),
            AttentionUpBlock(64, 32, 32)
        ])

        # Output
        self.outc = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        skips = []
        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        x = self.bottleneck(x)

        for layer, skip in zip(self.decoder, reversed(skips)):
            x = layer(x, skip)

        return self.outc(x)

In [ ]:
class SoftDiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.4, dice_weight=0.6, smooth=1e-7):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        
        probs = torch.sigmoid(logits)
        probs_flat = probs.flatten(start_dim=2)
        targets_flat = targets.flatten(start_dim=2)
        
        intersection = (probs_flat * targets_flat).sum(dim=2)
        cardinality = probs_flat.sum(dim=2) + targets_flat.sum(dim=2)
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss = 1.0 - dice_score.mean()
        
        return (self.bce_weight * bce_loss) + (self.dice_weight * dice_loss)


@torch.no_grad()
def calculate_competition_metrics(predictions, targets, threshold=None, min_defect_pixels=None, eps=1e-7):
    if threshold is None:
        threshold = CONFIG["class_thresholds"]
    if min_defect_pixels is None:
        min_defect_pixels = CONFIG["min_defect_pixels"]

    # Convert thresholds to tensor
    if isinstance(threshold, (list, tuple)):
        threshold = torch.tensor(threshold, device=predictions.device, dtype=predictions.dtype).view(1, -1, 1, 1)

    probs = torch.sigmoid(predictions)
    pred_masks = (probs >= threshold).float()
    targets = (targets > 0.5).float()

    # Minimum pixel filtering
    pred_pixels = pred_masks.sum(dim=(2, 3), keepdim=True)
    pred_masks = torch.where(pred_pixels >= min_defect_pixels, pred_masks, torch.zeros_like(pred_masks))

    pred_flat = pred_masks.flatten(start_dim=2)
    target_flat = targets.flatten(start_dim=2)

    pred_pixels = pred_flat.sum(dim=2)
    target_pixels = target_flat.sum(dim=2)
    intersection = (pred_flat * target_flat).sum(dim=2)
    total_pixels = targets.shape[-2] * targets.shape[-1]

    positive_pairs = target_pixels > 0
    empty_pairs = ~positive_pairs

    dice_per_pair = (2 * intersection + eps) / (pred_pixels + target_pixels + eps)
    foreground_dice = dice_per_pair[positive_pairs].mean().item() if positive_pairs.any() else 0.0

    fp_fraction = pred_pixels / total_pixels
    fp_score = 1.0 - fp_fraction[empty_pairs].mean().item() if empty_pairs.any() else 1.0

    correctly_empty = pred_pixels == 0
    empty_accuracy = correctly_empty[empty_pairs].float().mean().item() if empty_pairs.any() else 1.0

    score = 0.70 * foreground_dice + 0.20 * fp_score + 0.10 * empty_accuracy

    return {"score": score, "foreground_dice": foreground_dice, "false_positive_score": fp_score, "empty_accuracy": empty_accuracy}

@torch.no_grad()
def calculate_filtered_metrics(pred_masks, targets, eps=1e-7):
    pred_masks = pred_masks.float().cpu()
    targets = (targets.cpu() > 0.5).float()

    pred_flat = pred_masks.flatten(start_dim=2)
    target_flat = targets.flatten(start_dim=2)

    pred_pixels = pred_flat.sum(dim=2)
    target_pixels = target_flat.sum(dim=2)
    intersection = (pred_flat * target_flat).sum(dim=2)
    total_pixels = targets.shape[-2] * targets.shape[-1]

    positive_pairs = target_pixels > 0
    empty_pairs = ~positive_pairs

    dice_per_pair = (2 * intersection + eps) / (pred_pixels + target_pixels + eps)
    foreground_dice = dice_per_pair[positive_pairs].mean().item() if positive_pairs.any() else 0.0

    fp_fraction = pred_pixels / total_pixels
    fp_score = 1.0 - fp_fraction[empty_pairs].mean().item() if empty_pairs.any() else 1.0

    correctly_empty = pred_pixels == 0
    empty_accuracy = correctly_empty[empty_pairs].float().mean().item() if empty_pairs.any() else 1.0

    score = 0.70 * foreground_dice + 0.20 * fp_score + 0.10 * empty_accuracy

    return {"score": score, "foreground_dice": foreground_dice, "false_positive_score": fp_score, "empty_accuracy": empty_accuracy}

In [ ]:
import time
# 1. Automatic Mixed Precision
def get_amp():
    if device.type == "cuda":
        scaler = torch.amp.GradScaler("cuda")
        autocast = lambda: torch.amp.autocast("cuda")
    else:
        scaler = None
        autocast = nullcontext
    return scaler, autocast


# 2. Training
def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device, log_every=1, desc="Training"):
    model.train()
    total_loss, total_samples = 0.0, 0

    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(dataloader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for batch_idx, (images, targets) in enumerate(pbar, start=1):
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        batch_size = images.size(0)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

        total_loss += loss.detach().item() * batch_size
        total_samples += batch_size

        batch_metrics = calculate_competition_metrics(logits.detach(), targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        if batch_idx % log_every == 0 or batch_idx == len(dataloader):
            avg_loss = total_loss / total_samples
            avg_metrics = {k: v / metric_count for k, v in metrics_sum.items()}

            pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}", 
                               "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}" })

        del logits, loss

    epoch_metrics = {k: v / metric_count for k, v in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics

def apply_prediction_filters(probs):
    thresholds = torch.tensor(CONFIG["class_thresholds"], device=probs.device).view(1, -1, 1, 1)

    # Convert probabilities into binary masks
    masks = (probs >= thresholds).byte().cpu().numpy()
    filtered_masks = np.zeros_like(masks, dtype=np.uint8)

    min_component = CONFIG["min_component_pixels"]
    min_pixels = CONFIG["min_defect_pixels"]

    for b in range(masks.shape[0]):
        for c in range(masks.shape[1]):
            mask = masks[b, c]

            # Connected component filtering
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            filtered = np.zeros_like(mask)

            for i in range(1, num_labels):
                area = stats[i, cv2.CC_STAT_AREA]
                if area >= min_component:
                    filtered[labels == i] = 1

            # Whole-mask minimum pixel filtering
            if filtered.sum() >= min_pixels:
                filtered_masks[b, c] = filtered

    return torch.from_numpy(filtered_masks).to(probs.device)
    
@torch.inference_mode()
def predict(model, images, device):
    
    model.eval()
    images = images.to(device, non_blocking=True)

    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16): logits = model(images)
            
    else: logits = model(images)
    return logits

# 3. Validation
@torch.inference_mode()
def validate_epoch(model, dataloader, criterion, device, log_every=1, desc="Validation"):
    
    model.eval()
    total_loss, total_samples = 0.0, 0

    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(dataloader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for batch_idx, (images, targets) in enumerate(pbar, start=1):
        targets = targets.to(device, non_blocking=True)
        batch_size = images.size(0)

        logits = predict(model, images, device)
        loss = criterion(logits, targets)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        batch_metrics = calculate_competition_metrics(logits, targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        if batch_idx % log_every == 0 or batch_idx == len(dataloader):
            avg_loss = total_loss / total_samples
            avg_metrics = {k: v / metric_count for k, v in metrics_sum.items()}

            pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}",
                "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}" })

        del logits, loss

    epoch_metrics = {k: v / metric_count for k, v in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics


@torch.inference_mode()
def validate_epoch_filtered(model, loader, criterion, device, desc="Filtered Validation"):
    model.eval()
    total_loss, total_samples = 0.0, 0
    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(loader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for images, targets in pbar:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        batch_size = images.size(0)

        logits = predict(model, images, device)

        # Calculate loss using original logits
        loss = criterion(logits, targets)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # Apply prediction filters
        probs = torch.sigmoid(logits)
        pred_masks = apply_prediction_filters(probs)

        # Calculate filtered metrics
        batch_metrics = calculate_filtered_metrics(pred_masks, targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        # Calculate average metrics
        avg_loss = total_loss / total_samples
        avg_metrics = {key: value / metric_count for key, value in metrics_sum.items()}

        pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}",
            "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}"
        })

    epoch_metrics = {key: value / metric_count for key, value in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics

# 4. Complete Training Function
def fit_model(model, train_loader, valid_loader, optimizer, criterion, epochs, log_every=1, run_name=None, architecture="U-Net", save_path=None):
    
    if num_gpus > 1 and device.type == "cuda": model = torch.nn.DataParallel(model)
    scaler, autocast = get_amp()
    
    print(f"Starting Model Training.\nUsing Device: {device} | Available GPUs: {num_gpus}\n")


    metric_names = ["loss", "score", "foreground_dice", "false_positive_score", "empty_accuracy"]
    history = { "train": {name: [] for name in metric_names}, "valid": {name: [] for name in metric_names} }

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-7 )

    # Early stopping settings
    best_score = float("-inf")
    early_stop_counter = 0
    patience = 5
    min_delta = 0.0001

    t_fit = time.time()
    for epoch in range(epochs):
        e0 = time.time()
        train_metrics = train_one_epoch( model, train_loader, optimizer, criterion, scaler, device, log_every=log_every, desc=f"[Train] [{epoch + 1}/{epochs}]" )
        # valid_metrics = validate_epoch( model, valid_loader, criterion, device, log_every=1, desc=f"[Valid] [{epoch + 1}/{epochs}]" )
        valid_metrics = validate_epoch_filtered( model, valid_loader, criterion, device, desc=f"[Valid] [{epoch + 1}/{epochs}]"  )

        for name in metric_names:
            history["train"][name].append(train_metrics[name])
            history["valid"][name].append(valid_metrics[name])

        
        # Update scheduler
        current_score = valid_metrics["score"]
        scheduler.step(current_score)

        # Save best model: architecture + weights
        if current_score > best_score + min_delta:
        
            best_score = current_score
            early_stop_counter = 0
        
            model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
            torch.save(model_to_save, save_path)
            print(f"Best model saved: {save_path}")
        
        else: early_stop_counter += 1
            
        el = time.time() - t_fit
        print(f"epoch {epoch+1}/{epochs} | val score {current_score:.4f} best {best_score:.4f} | {el:.0f}s elapsed, ~{el/(epoch+1)*(epochs-epoch-1):.0f}s left", flush=True)

        gc.collect()

        # Stop training when validation score stops improving
        if early_stop_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break
        
    return history

In [ ]:

# ===== uint8 preload (1.2GB images instead of 4.7GB float32) =====
df = pd.read_csv(CSV_PATH)
H, W = int(df["height"].iloc[0]), int(df["width"].iloc[0])
n = len(X)
images_u8 = np.empty((n, H, W), dtype=np.uint8)
masks = np.zeros((n, 4, H, W), dtype=np.uint8)
groups = {str(i): g for i, g in df.groupby("image_id")}
for i, image_id in enumerate(tqdm(X, desc="preload", dynamic_ncols=True)):
    iid = str(image_id)
    name = iid if Path(iid).suffix else f"{iid}.jpg"
    with Image.open(TRAIN_DIR / name) as im:
        a = np.asarray(im.convert("L"), dtype=np.uint8)
    if a.shape != (H, W):
        raise ValueError(f"{iid} shape {a.shape}")
    images_u8[i] = a
    g = groups.get(iid)
    if g is not None:
        for row in g.itertuples(index=False):
            if pd.isna(row.mask_rle) or not str(row.mask_rle).strip():
                continue
            c = CLASS_TO_IDX[row.defect_type]
            masks[i, c] = np.maximum(masks[i, c], rle_decode(row.mask_rle, H, W))
print(f"preload done: images {images_u8.nbytes/1e9:.2f}GB | masks {masks.nbytes/1e9:.2f}GB", flush=True)

def augment(image, mask):
    # image (1,H,W) float32, mask (C,H,W) uint8
    if random.random() < 0.5:
        image = image[:, :, ::-1]; mask = mask[:, :, ::-1]
    if random.random() < 0.25:
        image = image[:, ::-1, :]; mask = mask[:, ::-1, :]
    if random.random() < 0.3:
        image = image * (0.9 + 0.2 * random.random()) + (random.random() - 0.5) * 0.05
        image = np.clip(image, 0.0, 1.0)
    return np.ascontiguousarray(image), np.ascontiguousarray(mask)



class SurfaceDefectDataset(Dataset):
    def __init__(self, images_u8, masks, is_train=False):
        self.images, self.masks, self.is_train = images_u8, masks, is_train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].astype(np.float32) / 255.0
        image = image[None]  # (1,H,W)
        mask = self.masks[idx]
        if self.is_train:
            image, mask = augment(image, mask)
        return torch.from_numpy(np.ascontiguousarray(image)), torch.from_numpy(mask.astype(np.float32))


In [ ]:
# ===== fine-tune fold 1 for 12 extra epochs (lr 3e-4) + test probs =====
t_start = time.time()
TIME_BUDGET = 2.5 * 3600
EPOCHS = 12
FINETUNE_FROM = next(Path("/kaggle/input").rglob("ATTENTION-UNET-FOLD1.pth"))
SAMPLE = ROOT / "sample_submission.csv"
print("fine-tune checkpoint:", FINETUNE_FROM, flush=True)
kfold = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["seed"])
fold_splits = list(kfold.split(X, y))
base_train = SurfaceDefectDataset(images_u8, masks, is_train=True)
base_val = SurfaceDefectDataset(images_u8, masks, is_train=False)
val_scores = {}

for fold in (1,):
    tr_idx, va_idx = fold_splits[fold - 1]
    print(f"STAGE fold {fold}/5 | train {len(tr_idx)} val {len(va_idx)} | {time.time()-t_start:.0f}s", flush=True)
    tr_loader = DataLoader(Subset(base_train, tr_idx), batch_size=64, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    va_loader = DataLoader(Subset(base_val, va_idx), batch_size=64, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    model = torch.load(FINETUNE_FROM, map_location=device, weights_only=False).to(device)
    print("loaded checkpoint:", type(model).__name__, flush=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=CONFIG["weight_decay"])
    criterion = SoftDiceBCELoss(bce_weight=0.4, dice_weight=0.6)
    save_path = "/kaggle/working/ATTENTION-UNET-FOLD1FT.pth"
    history = fit_model(model, tr_loader, va_loader, optimizer, criterion,
                        epochs=EPOCHS, log_every=40, run_name=f"ATTN-FOLD{fold}FT", save_path=save_path)
    val_scores[f"fold{fold}ft"] = max(history["valid"]["score"])
    print(f"fold {fold}ft best val {val_scores[f'fold{fold}ft']:.4f} | {time.time()-t_start:.0f}s elapsed", flush=True)
    json.dump(val_scores, open("/kaggle/working/fold_scores.json", "w"))
    del model, optimizer, tr_loader, va_loader
    gc.collect(); torch.cuda.empty_cache()

# ===== test probs from the fine-tuned checkpoint (hflip TTA, same as infer-v4) =====
H, W = 128, 800
TEST_DIR = ROOT / "test"
test_ids = pd.read_csv(SAMPLE)["image_id"].unique()
test_u8 = np.empty((len(test_ids), H, W), dtype=np.uint8)
for i, iid in enumerate(tqdm(test_ids, desc="test imgs", dynamic_ncols=True)):
    name = iid if Path(iid).suffix else f"{iid}.jpg"
    with Image.open(TEST_DIR / name) as im:
        test_u8[i] = np.asarray(im.convert("L"), dtype=np.uint8)
print(f"test_u8 {test_u8.nbytes/1e9:.2f}GB | {time.time()-t_start:.0f}s", flush=True)

model = torch.load(save_path, map_location=device, weights_only=False).to(device).eval()
out = np.empty((len(test_u8), 4, H, W), dtype=np.float16)
t0 = time.time()
with torch.inference_mode():
    for s in range(0, len(test_u8), 64):
        x = torch.from_numpy(test_u8[s:s + 64]).unsqueeze(1).float().div_(255).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            logits = (logits + torch.flip(model(torch.flip(x, dims=[-1])), dims=[-1])) / 2
        out[s:s + 64] = torch.sigmoid(logits).float().cpu().numpy().astype(np.float16)
        if (s // 64) % 20 == 0:
            el = time.time() - t0
            print(f"  test {min(s+64,len(test_u8))}/{len(test_u8)} | {el:.0f}s | ETA {el/(s+64)*(len(test_u8)-s-64):.0f}s", flush=True)
np.save("/kaggle/working/test_probs_fold1.npy", out)
print(f"fold1ft probs saved | mean {out.mean():.4f} | {time.time()-t_start:.0f}s", flush=True)
print("TRAINING DONE:", val_scores, flush=True)
